# 08. Validation 없는 시간순 Train/Test 8:2 근사 실험 (Legacy 지정가)

> **사용 중단:** 이 notebook은 NO_FILL 지정가 라벨을 사용한 비교용이다. 현재 primary는 즉시체결 3-outcome인 `09_immediate_fill_rebaseline.ipynb`다.

9개 날짜를 쪼개지 않고 앞 7개 세션을 Train, 뒤 2개 세션을 Test로 사용한다. 행 기준 비율은 약 82.6:17.4다. 기존 Reduced V2에서 확정한 90개 feature와 3,044-parameter 모델을 사용하며 validation, early stopping, temperature calibration은 모두 제거한다.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENT.md').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fixed_train_test import run_fixed_train_test_experiment

assert 'envs/urban' in str(Path(sys.executable).resolve()), sys.executable
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 120)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'python: {sys.executable}')

PROJECT_ROOT: /home/user/urbandatalab/YSLee/code/Detecting-entry-signals-for-short-term-trades-right-before-a-rapid-price-surge
python: /home/user/anaconda3/envs/urban/bin/python


## 1. 고정 15 epoch 학습

Train은 세션·종목·10분 bucket당 한 행만 사용하고 날짜별 총 loss weight를 동일하게 둔다. Test에는 학습이나 threshold 재조정을 하지 않는다.

In [2]:
result = run_fixed_train_test_experiment(PROJECT_ROOT)
print(f"device: {result['device']}")
print(f"features: {result['feature_count']}, parameters: {result['parameter_count']:,}")
display(result['sampling_balance'])
display(result['training_history'])

device: cuda
features: 90, parameters: 3,044


,session,source_rows,is_train,sampled_rows,total_loss_weight
0,session_2026-07-07,2677,True,282,603.714294
1,session_2026-07-08,6991,True,735,603.714294
2,session_2026-07-09,6699,True,695,603.714294
3,session_2026-07-10,6859,True,718,603.714294
4,session_2026-07-13,5959,True,616,603.714294
5,session_2026-07-14,4647,True,489,603.714294
6,session_2026-07-15,6629,True,691,603.714294
7,session_2026-07-16,4384,False,0,0.000000
8,session_2026-07-17,4167,False,0,0.000000


,epoch,train_cross_entropy
0,1,1.109073
1,2,1.064625
2,3,1.044145
3,4,1.036570
4,5,1.022582
5,6,1.019805
6,7,1.010909
7,8,1.001459
8,9,0.995984
9,10,0.991430


## 2. Train–Test 일반화 격차

Train 지표는 학습에 사용한 날짜를 다시 예측한 in-sample 결과다. 따라서 Test보다 좋아 보이는 정도를 과적합 진단용으로만 비교한다.

In [3]:
display(result['metrics'])
display(result['generalization_gap'])

,evaluation_group,session,samples,multiclass_logloss,multiclass_brier,accuracy,macro_pr_auc,tp_pr_auc,tp_roc_auc,tp_positive_rate,tp_ece,return_spearman,return_mae,top1_mean_actual_return,top5_mean_actual_return
0,train,ALL,40461,0.977951,0.544497,0.588764,0.405689,0.171868,0.797114,0.055140,0.014352,0.156018,0.010793,-0.000358,-0.000605
1,test,ALL,8551,0.858794,0.475853,0.668928,0.381123,0.122113,0.812359,0.035434,0.017569,0.129636,0.007902,-0.001504,-0.000494
2,train,session_2026-07-07,2677,0.796809,0.447607,0.680986,0.402770,0.088424,0.827666,0.025028,0.015893,0.180946,0.006647,-0.001254,-0.000797
3,train,session_2026-07-08,6991,0.959203,0.519923,0.623230,0.378756,0.125602,0.762254,0.049206,0.015020,0.157979,0.010550,-0.000028,-0.000338
4,train,session_2026-07-09,6699,0.854757,0.468999,0.680549,0.401959,0.178405,0.817763,0.035229,0.020828,0.131513,0.008037,0.002638,-0.000046
5,train,session_2026-07-10,6859,1.017815,0.561618,0.572095,0.408180,0.200355,0.769649,0.086456,0.013729,0.160054,0.013390,-0.001887,-0.000953
6,train,session_2026-07-13,5959,1.003769,0.568271,0.563182,0.410566,0.195446,0.800451,0.057728,0.013474,0.115817,0.010737,0.001738,-0.001447
7,train,session_2026-07-14,4647,0.994536,0.554723,0.560146,0.444809,0.175151,0.809555,0.056596,0.020751,0.219224,0.011729,-0.001466,-0.000174
8,train,session_2026-07-15,6629,1.119283,0.639578,0.482727,0.388323,0.161592,0.755506,0.057927,0.013816,0.128517,0.012217,-0.002473,-0.001323
9,test,session_2026-07-16,4384,0.921096,0.503500,0.631843,0.415266,0.134144,0.764545,0.049954,0.011657,0.174831,0.009978,-0.001896,-0.000450


,metric,train,test,test_minus_train
0,multiclass_logloss,0.977951,0.858794,-0.119156
1,macro_pr_auc,0.405689,0.381123,-0.024566
2,tp_pr_auc,0.171868,0.122113,-0.049754
3,tp_roc_auc,0.797114,0.812359,0.015244
4,return_spearman,0.156018,0.129636,-0.026382
5,return_mae,0.010793,0.007902,-0.002891
6,top1_mean_actual_return,-0.000358,-0.001504,-0.001146
7,top5_mean_actual_return,-0.000605,-0.000494,0.000110


## 3. Outcome을 사용하지 않는 threshold

Validation이 없으므로 수익률을 보고 threshold를 최적화하지 않는다. Train 예상수익 점수의 99% 분위수와 0% 중 큰 값을 고정한다. 이번 실행에서는 99% 분위수가 음수여서 0%가 적용됐다.

In [4]:
display(result['threshold'])
display(result['backtest_metrics'])
display(result['session_metrics'])

,threshold,method,train_score_quantile,uses_outcome_for_threshold_selection,temperature,validation_used
0,0.0,train_score_quantile_without_outcome_tuning,0.99,False,1.0,False


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group
0,112,77,103,68,26,29,13,0.382353,0.004320,-0.00162,0.004298,0.029195,291.946061,1.279748,-0.024529,9,0,0,0.0,10291.946061,train
1,8,7,8,4,0,2,2,0.000000,-0.018905,-0.02336,-0.018908,-0.007559,-75.588225,0.050922,-0.007964,0,0,0,0.0,9924.411775,test


,signals_above_threshold,ten_minute_signal_clusters,order_attempts,filled_trades,tp_trades,sl_trades,timeout_trades,tp_precision_given_fill,mean_net_return_per_fill,median_net_return_per_fill,net_return_on_deployed_capital,return_on_initial_capital,total_net_pnl,profit_factor,max_drawdown,skipped_same_symbol,skipped_position_limit,skipped_cash_limit,threshold,ending_cash,evaluation_group,session
0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,0.000000,0,0,0,0.0,10000.000000,train,session_2026-07-07
1,18,12,18,12,4,6,2,0.333333,0.000974,-0.016447,0.000963,0.001155,11.546210,1.058308,-0.015126,0,0,0,0.0,10011.546210,train,session_2026-07-08
2,11,8,10,5,4,1,0,0.800000,0.029115,0.046083,0.029068,0.014509,145.094203,4.629187,-0.003998,1,0,0,0.0,10145.094203,train,session_2026-07-09
3,35,21,29,22,8,10,4,0.363636,0.004389,0.003654,0.004374,0.009606,96.064337,1.283691,-0.011345,6,0,0,0.0,10096.064337,train,session_2026-07-10
4,16,13,15,9,5,2,2,0.555556,0.017818,0.046433,0.017797,0.015974,159.736528,2.924618,-0.003271,1,0,0,0.0,10159.736528,train,session_2026-07-13
5,12,10,12,6,2,2,2,0.333333,-0.003859,-0.024666,-0.003876,-0.002325,-23.245032,0.798672,-0.008895,0,0,0,0.0,9976.754968,train,session_2026-07-14
6,20,13,19,14,3,8,3,0.214286,-0.006948,-0.033029,-0.006943,-0.009725,-97.250185,0.637838,-0.016749,1,0,0,0.0,9902.749815,train,session_2026-07-15
7,4,3,4,3,0,2,1,0.000000,-0.026560,-0.032960,-0.026553,-0.007964,-79.643865,0.000000,-0.007964,0,0,0,0.0,9920.356135,test,session_2026-07-16
8,4,4,4,1,0,0,1,0.000000,0.004063,0.004063,0.004063,0.000406,4.055640,inf,0.000000,0,0,0,0.0,10004.055640,test,session_2026-07-17


## 4. 해석 제한

7/16~17은 이전 실험에서 이미 확인한 날짜이므로 pristine test가 아니다. 이 실험은 Train을 늘렸을 때의 상대 변화만 확인하며 배포 승인에는 사용하지 않는다.

In [5]:
test_row = result['backtest_metrics'].query("evaluation_group == 'test'").iloc[0]
display(Markdown(
    f"**Test 체결 {int(test_row['filled_trades'])}건, 평균 순수익률 {test_row['mean_net_return_per_fill']:.3%}.**  "
    "표본이 적고 비독립 test이므로 실전 성능으로 확정하지 않는다."
))

**Test 체결 4건, 평균 순수익률 -1.890%.**  표본이 적고 비독립 test이므로 실전 성능으로 확정하지 않는다.

## 5. 저장 artifact

In [6]:
display(pd.DataFrame({'artifact': result['paths'].keys(), 'path': map(str, result['paths'].values())}))

,artifact,path
0,predictions,/home/user/urbandatalab/YSLee/data/stock_data/...
1,metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
2,generalization_gap,/home/user/urbandatalab/YSLee/data/stock_data/...
3,training_history,/home/user/urbandatalab/YSLee/data/stock_data/...
4,sampling_balance,/home/user/urbandatalab/YSLee/data/stock_data/...
5,payoffs,/home/user/urbandatalab/YSLee/data/stock_data/...
6,threshold,/home/user/urbandatalab/YSLee/data/stock_data/...
7,backtest_metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
8,session_metrics,/home/user/urbandatalab/YSLee/data/stock_data/...
9,ledger,/home/user/urbandatalab/YSLee/data/stock_data/...
